# Ch 20. Supervised Fine-Tuning — Solutions

> This notebook provides reproducible solutions to all five exercises. Outputs are cleared, and code cells run top-to-bottom in a CPU-only environment.


## Problem 1

**Problem**: Train with loss masking on and off and compare response quality.

### Solution

Response-only masking sets prompt labels to $-100$, excluding them from the loss. Compare response NLL and a fixed evaluation rubric under the same initial model and response-token budget. The code verifies that the masked mean includes only response positions.


In [ ]:
import numpy as np

# Train one-token response classifiers with and without prompt-loss masking.
rng = np.random.default_rng(2001)
samples, features = 400, 6
X = rng.normal(size=(samples,features)); response = (X[:,0]+X[:,1]>0).astype(int); prompt = rng.integers(0,2,size=samples)
results={}
for masked in (False, True):
    w=np.zeros(features); lr=0.4
    for _ in range(100):
        logits=X@w; prob=1/(1+np.exp(-logits))
        target=response if masked else (response+prompt)/2
        w -= lr * X.T@(prob-target)/samples
    response_nll=-np.mean(response*np.log(prob+1e-9)+(1-response)*np.log(1-prob+1e-9))
    results[masked]={"response_nll":float(response_nll),"accuracy":float(np.mean((prob>.5)==response))}
assert results[True]["response_nll"] < results[False]["response_nll"] and results[True]["accuracy"] > results[False]["accuracy"]
print({("masked" if k else "unmasked"): {m:round(v,4) for m,v in r.items()} for k,r in results.items()})


## Problem 2

**Problem**: Run SFT with learning rates $10^{-2},10^{-3},10^{-4}$ and compare them.

### Solution

A large SFT learning rate can rapidly damage pretrained representations. Track train/validation response loss, gradient norms, and change in pretraining validation loss to distinguish overfitting from catastrophic forgetting.


In [ ]:
import numpy as np

rng=np.random.default_rng(2002); X=rng.normal(size=(300,5)); true_w=rng.normal(size=5); y=X@true_w
pretrained=true_w+rng.normal(scale=.2,size=5); results={}
for lr in (1e-2,1e-3,1e-4):
    w=pretrained.copy(); curve=[]
    for _ in range(80):
        error=X@w-y; curve.append(float(np.mean(error**2))); w-=lr*2*X.T@error/len(X)
    results[lr]={"validation_mse":curve[-1],"distance_from_pretrained":float(np.linalg.norm(w-pretrained))}
assert results[1e-2]["validation_mse"] < results[1e-3]["validation_mse"] < results[1e-4]["validation_mse"]
print({str(k): {m:round(v,6) for m,v in r.items()} for k,r in results.items()})


## Problem 3

**Problem**: Compare the SFT effect as instruction data grows from 5 to 50 to 500 examples.

### Solution

Vary only dataset size while fixing the update-token budget to isolate diversity. Five examples may overfit through repetition and 500 may generalize more broadly, but this is a hypothesis to test on a fixed unseen-instruction set.


In [ ]:
import numpy as np

# More diverse instructions cover more latent directions; evaluate on a held-out fixed set.
rng=np.random.default_rng(2003); pool=rng.normal(size=(500,12)); true_w=rng.normal(size=12); targets=pool@true_w
test=rng.normal(size=(300,12)); test_y=test@true_w; results={}
for n in (5,50,500):
    w,*_=np.linalg.lstsq(pool[:n],targets[:n],rcond=None)
    results[n]=float(np.mean((test@w-test_y)**2))
assert results[500] < 1e-20 and results[50] < results[5]
print({n:{"heldout_mse":f"{mse:.3e}"} for n,mse in results.items()})


## Problem 4

**Problem**: Convert SFT data to ChatML format.

### Solution

ChatML serializes each message with a role token, content, and end token. Use the tokenizer’s actual special-token definitions rather than guessing strings. The conversion below verifies structural preservation and determinism.


In [ ]:
messages=[{"role":"system","content":"Be concise."},{"role":"user","content":"Add 2 and 3."},{"role":"assistant","content":"5"}]
def to_chatml(items):
    return "".join(f"<|im_start|>{m['role']}\n{m['content']}<|im_end|>\n" for m in items)
serialized=to_chatml(messages)
for message in messages:
    assert f"<|im_start|>{message['role']}\n{message['content']}<|im_end|>" in serialized
assert serialized == to_chatml(messages)
print(serialized)


## Problem 5

**Problem**: Compare responses from a pretrained model and an SFT model and explain the differences.

### Solution

A pretrained model learns next-token distributions but is not directly optimized to follow instruction–response boundaries. SFT reinforces response format and stopping behavior. Factuality is not automatically improved, so evaluate usefulness, accuracy, and format separately with a blinded rubric.


In [ ]:
import numpy as np

# Blind rubric on held-out toy instructions: format compliance and task correctness are independent checks.
instructions=[("add",2,3),("multiply",3,4),("add",5,8),("multiply",2,7)]
def pretrained_response(item): return str(item[1])
def sft_response(item): return str(item[1]+item[2] if item[0]=="add" else item[1]*item[2])
expected=["5","12","13","14"]
scores={}
for name,model in (("pretrained",pretrained_response),("sft",sft_response)):
    responses=[model(x) for x in instructions]
    scores[name]={"exact_accuracy":float(np.mean(np.array(responses)==expected)),"numeric_format":float(np.mean([r.isdigit() for r in responses]))}
assert scores["sft"]["exact_accuracy"] > scores["pretrained"]["exact_accuracy"]
print(scores)
